**Sample ID**: 808




**Query**: Monitor recent messages in the Slack channel “ABP-project-progress” for any updates indicating a project cancellation. When such a cancellation is detected, locate the corresponding file named "ABP_Pending_Project_Proposal.docx" in Google Drive and move it to the "Rejected Proposals" folder. Finally, based on the outcome of the review, post a message in the 'project-progress' channel indicating "project cancelled" / "project continues".




**DB Type**: Edge Case 2




**Case Description**: A Slack message indicating a project cancellation exists. The “ABP_Pending_Project_Proposal” file exists in Google Drive and has been already moved to “Rejected Proposals” folder. Both Slack channels, ABP-project-progress and project-progress, are present.




**Global/Context Variables**:


- abp_project_progress_channel_name = "ABP-project-progress"
- project_progress_channel_name = "project-progress"
- proposal_file_name = "ABP_Pending_Project_Proposal.docx"
- rejected_proposals_folder_name = "Rejected Proposals"
- cancelled_message = "project cancelled"
- not_cancelled_message = "project continues"




**APIs**:

- slack
- gdrive


# Set Up

## Download relevant files

In [28]:
import io
import os
import sys
import zipfile
import shutil
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Version to download
VERSION = "0.1.4"  # Version of the API

# Define paths
CONTENT_DIR = '/content'
APIS_DIR = os.path.join(CONTENT_DIR, 'APIs')
DBS_DIR = os.path.join(CONTENT_DIR, 'DBs')
SCRIPTS_DIR = os.path.join(CONTENT_DIR, 'Scripts')
FC_DIR = os.path.join(CONTENT_DIR, 'Schemas')
ZIP_PATH = os.path.join(CONTENT_DIR, f'APIs_V{VERSION}.zip')

# Google Drive Folder ID where versioned APIs zip files are stored
APIS_FOLDER_ID = '1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4'

# List of items to extract from the zip file
ITEMS_TO_EXTRACT = ['APIs/', 'DBs/', 'Scripts/', 'Schemas/']

# Clean up existing directories and files
for path in [APIS_DIR, DBS_DIR, SCRIPTS_DIR, FC_DIR, ZIP_PATH]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Authenticate and create the drive service
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Helper function to download a file from Google Drive
def download_drive_file(service, file_id, output_path, file_name=None, show_progress=True):
    """Downloads a file from Google Drive"""
    destination = output_path
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if show_progress:
                print(f"Download progress: {int(status.progress() * 100)}%")


# 1. List files in the specified APIs folder
print(f"Searching for APIs zip file with version {VERSION} in folder: {APIS_FOLDER_ID}...")
apis_file_id = None

try:
    query = f"'{APIS_FOLDER_ID}' in parents and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    for file in files:
        file_name = file.get('name', '')
        if file_name.lower() == f'apis_v{VERSION.lower()}.zip':
            apis_file_id = file.get('id')
            print(f"Found matching file: {file_name} (ID: {apis_file_id})")
            break

except Exception as e:
    print(f"An error occurred while listing files in Google Drive: {e}")

if not apis_file_id:
    print(f"Error: Could not find APIs zip file with version {VERSION} in the specified folder.")
    sys.exit("Required APIs zip file not found.")

# 2. Download the found APIs zip file
print(f"Downloading APIs zip file with ID: {apis_file_id}...")
download_drive_file(drive_service, apis_file_id, ZIP_PATH, file_name=f'APIs_V{VERSION}.zip')

# 3. Extract specific items from the zip file to /content
print(f"Extracting specific items from {ZIP_PATH} to {CONTENT_DIR}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()

        for member in zip_contents:
            extracted = False
            for item_prefix in ITEMS_TO_EXTRACT:
              if member == item_prefix or member.startswith(item_prefix):
                    zip_ref.extract(member, CONTENT_DIR)
                    extracted = True
                    break

except zipfile.BadZipFile:
    print(f"Error: The downloaded file at {ZIP_PATH} is not a valid zip file.")
    sys.exit("Invalid zip file downloaded.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")
    sys.exit("Extraction failed.")


# 4. Clean up
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 5. Add APIs to path
if os.path.exists(APIS_DIR):
    sys.path.append(APIS_DIR)
else:
    print(f"Error: APIS directory not found at {APIS_DIR} after extraction. Cannot add to path.")

# 6. Quick verification
# Check for the presence of the extracted items
verification_paths = [APIS_DIR, DBS_DIR, SCRIPTS_DIR]
all_present = True
print("\nVerifying extracted items:")
for path in verification_paths:
    if os.path.exists(path):
        print(f"✅ {path} is present.")
    else:
        print(f"❌ {path} is MISSING!")
        all_present = False

if all_present:
    print(f"\n✅ Setup complete! Required items extracted to {CONTENT_DIR}.")
else:
    print("\n❌ Setup failed! Not all required items were extracted.")

# 7. Generate Schemas

print("\nGenerating FC Schemas")

# Change working directory to the source folder

# Iterate through the packages in the /content/APIs directory

    # Check if it's a directory (to avoid processing files)
        # Call the function to generate schema for the current package
print(f"✅ Successfully generated {len(os.listdir(FC_DIR))} FC Schemas to {FC_DIR}")
os.chdir(CONTENT_DIR)

Searching for APIs zip file with version 0.1.4 in folder: 1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4...
Found matching file: APIs_V0.1.4.zip (ID: 1TnAaWGfVrMxWTilyhy46-Aue_bh0XkNk)
Download progress: 100%
Extracting specific items from /content/APIs_V0.1.4.zip to /content...

Verifying extracted items:
✅ /content/APIs is present.
✅ /content/DBs is present.
✅ /content/Scripts is present.

✅ Setup complete! Required items extracted to /content.

Generating FC Schemas
✅ Successfully generated 70 FC Schemas to /content/Schemas


## Install Dependencies and Clone Repositories

In [29]:
!pip install -r /content/APIs/requirements.txt

## Import APIs and initiate DBs

In [30]:
import slack
import gdrive

slack.SimulationEngine.db.load_state('/content/DBs/SlackDefaultDB.json')
gdrive.SimulationEngine.db.load_state('/content/DBs/GDriveDefaultDB.json')

channel_name = "ABP-project-progress"
create_response = slack.create_channel(name=channel_name)
# Create required Slack channels if not already created
abp_channel_id = create_response["channel"]["id"]

resp2 = slack.create_channel(name="project-progress")
project_channel_id = resp2["channel"]["id"]

# Invite a user and post a cancellation message in ABP-project-progress
slack.post_chat_message(channel=abp_channel_id, text="Unfortunately, the project has been cancelled.")

rejected_folder = gdrive.create_file_or_folder({
    "name": "Rejected Proposals",
    "mimeType": "application/vnd.google-apps.folder"
})

folder_id=rejected_folder['id']
file_resp = gdrive.create_file_or_folder({
    "name": "ABP_Pending_Project_Proposal.docx",
    "mimeType": "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
    "parents": [folder_id]
})

# Initial Assertion
1. Assert that the "ABP-project-progress" Slack channel exists.
2. Assert that at least one recent message in the "ABP-project-progress" Slack channel indicates a project cancellation.
3. Assert that a file named "ABP_Pending_Project_Proposal.docx" exists in Google Drive.
4. Assert that the folder "Rejected Proposals"  exists in the drive.
5. Assert that "ABP_Pending_Project_Proposal.docx" file has already been moved to the "Rejected Proposals" folder.
6. Assert that the "project-progress" Slack channel exists.

In [31]:
from Scripts.assertions_utils import *
import slack
import gdrive
import re

# Constants
abp_project_progress_channel_name = "ABP-project-progress"
project_progress_channel_name = "project-progress"
proposal_file_name = "ABP_Pending_Project_Proposal.docx"
rejected_proposals_folder_name = "Rejected Proposals"

# local variables
cancellation_keywords = ["cancel", "cancelled", "rejected", "terminated"]

# 1. Assert that the "ABP-project-progress" Slack channel exists.
channel_list_response = slack.list_channels(types='public_channel,private_channel')
incident_channels = [ch for ch in channel_list_response.get('channels', []) if compare_strings(ch.get('name'), abp_project_progress_channel_name)]
assert len(incident_channels) == 1, f"Expected exactly one channel named '{abp_project_progress_channel_name}', but found {len(incident_channels)}."
channel_id = incident_channels[0].get('id') # Store ID for next assertion

# 2. Assert that at least one recent message in the "ABP-project-progress" Slack channel indicates a project cancellation.
history_response = slack.get_conversation_history(channel=channel_id)

cancellation_message_found = False
for message in history_response.get('messages', []):
    words_in_text = re.findall(r'\b[a-zA-Z]+\b', message.get('text', ''))
    if compare_is_list_subset(cancellation_keywords, words_in_text, list_comparison_function="any"):
        cancellation_message_found = True
        break

assert cancellation_message_found, f"Initial state check failed: No recent message indicating project cancellation found in channel '{abp_project_progress_channel_name}'."

# 3. Assert that a file named "ABP_Pending_Project_Proposal.docx" exists in Google Drive.
query=f"name = '{proposal_file_name}' and trashed = false and mimeType = 'application/vnd.openxmlformats-officedocument.wordprocessingml.document'"
response = gdrive.list_user_files(q=query,spaces='drive',supportsAllDrives=True,includeItemsFromAllDrives=True)
proposal_file_id = next((item.get("id") for item in response.get("files", []) if compare_strings(item.get("name"), proposal_file_name)),None)

assert proposal_file_id is not None, f"Initial state check failed: Google Drive file '{proposal_file_name}' not found."

# 4. Assert that the folder "Rejected Proposals"  exists in the drive.
folder_query=f"name = '{rejected_proposals_folder_name}' and trashed = false and mimeType = 'application/vnd.google-apps.folder'"
folder_response = gdrive.list_user_files(q=folder_query,spaces='drive',supportsAllDrives=True,includeItemsFromAllDrives=True)
rejected_folder_id = next((item.get("id") for item in folder_response.get("files", []) if compare_strings(item.get("name"), rejected_proposals_folder_name)),None)

# 5. Assert that "ABP_Pending_Project_Proposal.docx" file has already been moved to the "Rejected Proposals" folder.
file_info_response = gdrive.get_file_metadata_or_content(fileId=proposal_file_id)

file_parents = file_info_response.get('parents', [])
is_in_rejected_folder = rejected_folder_id in file_parents

assert is_in_rejected_folder, f"File '{proposal_file_name}' (ID:  is not in the '{rejected_proposals_folder_name}' folder ."

#  6. Assert that the "project-progress" Slack channel exists.
channel_list_response = slack.list_channels()
incident_channels = [ch for ch in channel_list_response.get('channels', []) if compare_strings(ch.get('name'), project_progress_channel_name)]
assert len(incident_channels) == 1, f"Expected exactly one channel named '{project_progress_channel_name}', but found {len(incident_channels)}."


# Action

- Monitor recent messages in the Slack channel “ABP-project-progress” for any updates indicating a project cancellation.
- When such a cancellation is detected, locate the corresponding file named "ABP_Pending_Project_Proposal.docx" in Google Drive and move it to the "Rejected Proposals" folder.
- Regardless of whether the project was canceled or not, post a message in the 'project-progress' channel indicating the outcome of the review, and confirm if the proposal file was moved in case of cancellation.

In [32]:
import time
import slack
import gdrive

# Constants
abp_project_progress_channel_name = "ABP-project-progress"
project_progress_channel_name = "project-progress"
proposal_file_name = "ABP_Pending_Project_Proposal.docx"
rejected_proposals_folder_name = "Rejected Proposals"
cancelled_message = "project cancelled"
not_cancelled_message = "project continues"


# Helper function to find Slack channel ID by name
def _find_slack_channel_id(slack_api_instance, channel_name):
    """Finds a Slack channel ID by its name."""
    channel_id = None
    cursor = None
    while True:
        response = slack.list_channels(
            types="public_channel,private_channel",
            limit=200,
            cursor=cursor
        )
        # Basic validation of response structure

        if not response.get('ok', True):
            print(f"Error: API call failed when listing Slack channels: {response.get('error', 'Unknown error')}")
            return None # Indicate failure
        if 'channels' not in response:
             print(f"Error: Unexpected response format (missing 'channels' list) when listing channels: {response}")
             return None

        for channel in response['channels']:
            if channel.get('name') == channel_name and 'id' in channel:
                channel_id = channel['id']
                break

        if channel_id:
            break

        cursor = response.get('response_metadata', {}).get('next_cursor')
        if not cursor:
            break

    return channel_id

# Helper function to find GDrive item ID by name and optional mimeType
def _find_gdrive_item_id(gdrive_api_instance, item_name, mime_type=None):
    """Finds a GDrive file or folder ID by its name and optionally mimeType."""
    item_id = None
    page_token = None
    query_parts = [f"name = '{item_name}'", "trashed = false"]
    if mime_type:
        query_parts.append(f"mimeType = '{mime_type}'")
    query = " and ".join(query_parts)

    while True:
        response = gdrive.list_user_files(
    q=query,
    spaces='drive',
    pageToken=page_token if page_token else '',
    supportsAllDrives=True,
    includeItemsFromAllDrives=True
)

        if 'files' not in response or not isinstance(response['files'], list):
             print(f"Error: Unexpected response format (missing 'files' list) when listing GDrive items for '{item_name}': {response}")
             return None

        for item in response['files']:
             if isinstance(item, dict) and item.get('name') == item_name and 'id' in item:
                item_id = item['id']
                return item_id

        page_token = response.get('nextPageToken')
        if not page_token:
            break

    return item_id

# ------------------- INSPECTION BLOCK -------------------
error_message = None

# 1. Find Slack channels
abp_channel_id = _find_slack_channel_id(slack, abp_project_progress_channel_name)
project_progress_channel_id = _find_slack_channel_id(slack, project_progress_channel_name)

# 2. Check recent messages in ABP channel
if not abp_channel_id:
    error_message = f"Could not find Slack channel '{abp_project_progress_channel_name}'."
else:
    print(f"Monitoring channel '{abp_project_progress_channel_name}' (ID: {abp_channel_id}) for cancellation messages...")
    history_response = slack.get_conversation_history(channel=abp_channel_id)

    if history_response.get('ok'):
        i = 0
        for message in history_response.get('messages', []):
          i+=1
          print(f"recent message # {i}: {message['text']}")


Monitoring channel 'ABP-project-progress' (ID: C53526390) for cancellation messages...
recent message # 1: Unfortunately, the project has been cancelled.


In [33]:
project_cancelled = True

file_moved = False
outcome_message = ""

# 3. Locate the file and folder
proposal_file_id = _find_gdrive_item_id(gdrive, proposal_file_name)
rejected_folder_id = _find_gdrive_item_id(gdrive, rejected_proposals_folder_name, mime_type='application/vnd.google-apps.folder')

if not proposal_file_id:
    print(f"❌ File '{proposal_file_name}' not found.")
if not rejected_folder_id:
    print(f"❌ Folder '{rejected_proposals_folder_name}' not found.")
if not project_progress_channel_id:
    print(f"❌ Target posting channel '{project_progress_channel_name}' not found.")

# ------------------- ACTION BLOCK -------------------
if project_cancelled and proposal_file_id and rejected_folder_id:
    print(f"Found file ID: {proposal_file_id}, Folder ID: {rejected_folder_id}. Getting current file location...")
    get_file_response = gdrive.get_file_metadata_or_content(fileId=proposal_file_id)

    if 'parents' in get_file_response:
        current_parents = get_file_response['parents']
        if rejected_folder_id in current_parents:
            print(f"File already in '{rejected_proposals_folder_name}'. No move needed.")
            file_moved = True
            outcome_message = f" Project ABP was cancelled. The proposal file '{proposal_file_name}' was already in the '{rejected_proposals_folder_name}' folder in Google Drive."
        elif not current_parents:
            print(f"Attempting to add file to folder '{rejected_proposals_folder_name}'...")
            update_response = gdrive.update_file_metadata_or_content(
                fileId=proposal_file_id,
                addParents=rejected_folder_id,

            )
            file_moved = 'id' in update_response
        else:
            parent_to_remove = current_parents[0]
            print(f"Moving file from parent '{parent_to_remove}' to '{rejected_proposals_folder_name}'...")
            update_response = gdrive.update_file_metadata_or_content(
                fileId=proposal_file_id,
                addParents=rejected_folder_id,
                removeParents=parent_to_remove,
            )
            if 'id' in update_response:
                time.sleep(2)
                verify_response = gdrive.get_file_metadata_or_content(fileId=proposal_file_id)

                file_moved = rejected_folder_id in verify_response.get('parents', [])
else:
    print("Skipping file move: Pre-conditions not met.")

if project_cancelled:
    if file_moved and not outcome_message:
        outcome_message = f" Project ABP was cancelled. The proposal file '{proposal_file_name}' has been moved to the '{rejected_proposals_folder_name}' folder in Google Drive."
    elif not file_moved:
        outcome_message = f" Project ABP was cancelled, but the proposal file '{proposal_file_name}' could *not* be moved."
else:
    outcome_message = "No project Cancellation was Detected"

if error_message and not project_cancelled:
    outcome_message += f" An error occurred during processing: {error_message}"

print(f"Outcome Message:{outcome_message}")
print(f"Posting outcome message to channel '{project_progress_channel_name}'...")
if project_progress_channel_id:
    response = slack.post_chat_message(
        channel=project_progress_channel_id,
        text=outcome_message
    )
    if not response.get('ok'):
        print(f" Failed to post message. Response: {response}")
    else:
        print(" Outcome message posted successfully.")
else:
    print(f"Outcome Message (not posted): {outcome_message}")

Found file ID: file_4, Folder ID: file_3. Getting current file location...
File already in 'Rejected Proposals'. No move needed.
Outcome Message: Project ABP was cancelled. The proposal file 'ABP_Pending_Project_Proposal.docx' was already in the 'Rejected Proposals' folder in Google Drive.
Posting outcome message to channel 'project-progress'...
 Outcome message posted successfully.


In [34]:
slack.list_channels(types="public_channel,private_channel", limit=100)


{'ok': True,
 'channels': [{'messages': [{'ts': '1688682784.334459',
     'user': 'U04L7NE5Q1Y',
     'text': "Welcome everyone to the marketing brainstorming session!  Let's kick off by sharing any initial campaign ideas for Q3.",
     'reactions': [{'name': 'rocket',
       'users': ['U04L7NE5Q1Y', 'U04M2R8JCQ6', 'U04M526DV51'],
       'count': 3}]},
    {'ts': '1688683000.456789',
     'user': 'U04M2R8JCQ6',
     'text': 'I think we should focus on a social media campaign highlighting our sustainability initiatives.',
     'reactions': [{'name': 'thumbsup',
       'users': ['U04L7NE5Q1Y', 'U04M526DV51', 'U04LMCYSD2X'],
       'count': 3}]},
    {'ts': '1688684000.987654',
     'user': 'U04LMCYSD2X',
     'text': 'Has anyone seen those interactive ads on platform X?',
     'reactions': []}],
   'conversations': {},
   'name': 'Default_Channel',
   'id': 'C04MKV1KQD6',
   'is_private': False,
   'team_id': None,
   'files': {'F04M89K2N': True, 'F04Pq7M9L': True}},
  {'messages': [{'ts

In [35]:
gdrive.list_user_files(q="name = 'ABP_Pending_Project_Proposal.docx'", pageSize=10)


{'kind': 'drive#fileList',
 'nextPageToken': None,
 'files': [{'kind': 'drive#file',
   'id': 'file_4',
   'driveId': '',
   'name': 'ABP_Pending_Project_Proposal.docx',
   'mimeType': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document',
   'parents': ['file_3'],
   'createdTime': '2025-03-14T00:00:00Z',
   'modifiedTime': '2025-03-14T00:00:00Z',
   'trashed': False,
   'starred': False,
   'owners': ['john.doe@gmail.com'],
   'size': '0',
   'md5Checksum': '',
   'sha1Checksum': '',
   'sha256Checksum': '',
   'imageMediaMetadata': {},
   'videoMediaMetadata': {},
   'permissions': [{'id': 'permission_file_4',
     'role': 'owner',
     'type': 'user',
     'emailAddress': 'john.doe@gmail.com'}],
   'enforceSingleParent': False,
   'ignoreDefaultVisibility': False,
   'keepRevisionForever': False,
   'ocrLanguage': '',
   'supportsAllDrives': False,
   'supportsTeamDrives': False,
   'useContentAsIndexableText': False,
   'includePermissionsForView': '',
   'incl

In [36]:
gdrive.list_user_files(q="name = 'Rejected Proposals' and mimeType = 'application/vnd.google-apps.folder'", pageSize=10)


{'kind': 'drive#fileList',
 'nextPageToken': None,
 'files': [{'kind': 'drive#file',
   'id': 'file_3',
   'driveId': '',
   'name': 'Rejected Proposals',
   'mimeType': 'application/vnd.google-apps.folder',
   'parents': [],
   'createdTime': '2025-03-14T00:00:00Z',
   'modifiedTime': '2025-03-14T00:00:00Z',
   'trashed': False,
   'starred': False,
   'owners': ['john.doe@gmail.com'],
   'size': '0',
   'md5Checksum': '',
   'sha1Checksum': '',
   'sha256Checksum': '',
   'imageMediaMetadata': {},
   'videoMediaMetadata': {},
   'permissions': [{'id': 'permission_file_3',
     'role': 'owner',
     'type': 'user',
     'emailAddress': 'john.doe@gmail.com'}],
   'enforceSingleParent': False,
   'ignoreDefaultVisibility': False,
   'keepRevisionForever': False,
   'ocrLanguage': '',
   'supportsAllDrives': False,
   'supportsTeamDrives': False,
   'useContentAsIndexableText': False,
   'includePermissionsForView': '',
   'includeLabels': '',
   'revisionSettings': {'keepForever': False

# Final Assertion
1. Assert that this file "ABP_Pending_Project_Proposal.docx" has been already moved to the "Rejected Proposals" folder.
2. Assert that the "project-progress" Slack channel contains message indicating the project has been cancelled.

In [37]:
from Scripts.assertions_utils import *
import slack
import gdrive
import re

proposal_file_name = "ABP_Pending_Project_Proposal.docx"
abp_project_progress_channel_name = "ABP-project-progress"
rejected_proposals_folder_name = "Rejected Proposals"
project_progress_channel_name = "project-progress"

# local variables
cancellation_keywords = ["cancel", "cancelled", "canceled", "rejected", "terminated",
                         "discontinued", "abandoned", "dropped", "halted"]

# 1. Assert that this file has moved to the "Rejected Proposals" folder.
query = (
    f"name = '{proposal_file_name}' and trashed = false and "
    "mimeType = 'application/vnd.openxmlformats-officedocument.wordprocessingml.document'"
)
response = gdrive.list_user_files(q=query, spaces='drive', supportsAllDrives=True, includeItemsFromAllDrives=True)
proposal_file_id = next(
    (item.get('id') for item in response.get('files', []) if compare_strings(item.get("name"), proposal_file_name)),
    None
)
assert proposal_file_id is not None, f"File '{proposal_file_name}' not found."

folder_query = (
    f"name = '{rejected_proposals_folder_name}' and trashed = false and "
    "mimeType = 'application/vnd.google-apps.folder'"
)
folder_response = gdrive.list_user_files(q=folder_query, spaces='drive', supportsAllDrives=True, includeItemsFromAllDrives=True)
rejected_folder_id = next(
    (item.get('id') for item in folder_response.get('files', []) if compare_strings(item.get('name'), rejected_proposals_folder_name)),
    None
)
assert rejected_folder_id is not None, f"Folder '{rejected_proposals_folder_name}' not found."

file_info_response = gdrive.get_file_metadata_or_content(fileId=proposal_file_id)
file_parents = file_info_response.get('parents', [])
is_in_rejected_folder = rejected_folder_id in file_parents
assert is_in_rejected_folder, f"File '{proposal_file_name}' is not in the '{rejected_proposals_folder_name}'"

# 2. Assert that the "project-progress" Slack channel contains a message indicating the project has been cancelled.
channel_list_response = slack.list_channels()
project_progress_id = next(
    (ch.get('id') for ch in channel_list_response.get('channels', []) if compare_strings(ch.get('name'), project_progress_channel_name)),
    None
)
assert project_progress_id is not None, f"Slack channel '{project_progress_channel_name}' not found."

history_response = slack.get_conversation_history(channel=project_progress_id, limit=20)
messages = history_response.get('messages', [])
found_project_cancel_msg = False
for msg in messages:
    words_in_text = [w.lower() for w in re.findall(r'\b[a-zA-Z]+\b', msg.get('text', ''))]
    if compare_is_list_subset(cancellation_keywords, words_in_text, list_comparison_function="any"):
        found_project_cancel_msg = True
        break

assert found_project_cancel_msg, f"No message in {project_progress_channel_name} stating project cancelled."
